In [ ]:
import os
import re
import pandas as pd
import numpy as np
import altair as alt


PATH_BASE = "./baselines_20260520_1943/app_bench_results.csv"
PATH_PARA = "./results_20260523_1603/app_bench_results.csv"

DIR_BASE_VTUNE = "./baselines_20260520_1943/vtune_csvs"
DIR_PARA_VTUNE = "./results_20260523_1603/vtune_csvs"


def parse_vtune_value(value_str: str) -> float:
    if not value_str: return 0.0
    try:
        cleaned = value_str.strip().replace(',', '').split()[0]
        return float(cleaned)
    except (ValueError, IndexError): return 0.0

def get_hotspot_function(file_path):
    if not os.path.exists(file_path): return "N/A", 0.0
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        for i, line in enumerate(lines):
            if "Function" in line and "Module" in line:
                target_line = lines[i+1]
                parts = [p.strip() for p in target_line.split('\t')]
                if len(parts) >= 4:
                    return parts[1], parse_vtune_value(parts[3])
        return "N/A", 0.0
    except: return "Error", 0.0

def get_metrics_from_csv(file_path: str, metrics_list: list) -> dict:
    results = {m: 0.0 for m in metrics_list}
    if not os.path.exists(file_path): return results
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = [p.strip() for p in line.split('\t')]
                for col_idx, part in enumerate(parts):
                    for m in metrics_list:
                        if part.lower() == m.lower():
                            for next_col in parts[col_idx + 1:]:
                                if next_col:
                                    results[m] = parse_vtune_value(next_col)
                                    break
    except Exception as e: pass
    return results

def get_hpc_metrics_isolated(file_path, metrics_list):
    results = {m: 0.0 for m in metrics_list}
    if not os.path.exists(file_path): return results
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for line in f:
                parts = [p.strip() for p in line.split('\t')]
                if len(parts) >= 3:
                    name_file = parts[1].lower()
                    value_file = parts[2]
                    for m in metrics_list:
                        if m.lower() == name_file:
                            results[m] = parse_vtune_value(value_file)
        return results
    except: return results

In [ ]:
def load_and_clean_data(path, is_baseline=False):
    df = pd.read_csv(path)

    # Identificação de O0 vs O3 para o baseline
    if is_baseline:
        df["opt_level"] = np.where(df.index % 6 < 3, "O0 (Raw)", "O3 (Opt)")
        df["threads"] = 1
        group_cols = ["dataset", "samples", "features", "threads", "opt_level"]
    else:
        group_cols = ["dataset", "samples", "features", "threads"]

    df_mean = df.groupby(group_cols).mean(numeric_only=True).reset_index()

    df_mean["problem_size"] = df_mean.apply(
        lambda x: f"{int(x['samples'])} x {int(x['features'])}", axis=1
    )
    
    # ATENÇÃO: Ordenando por 'features' primeiro, já que D é o gargalo aqui
    df_mean = df_mean.sort_values(['features', 'samples'])
    return df_mean

df_baseline = load_and_clean_data(PATH_BASE, is_baseline=True)
df_parallel = load_and_clean_data(PATH_PARA, is_baseline=False)

# Ordem correta para os gráficos do Altair
problem_order = df_baseline['problem_size'].unique().tolist()

print("✅ Dados do  carregados com sucesso!")
display(df_baseline[['problem_size', 'opt_level', 'fit_time']])

In [ ]:
# Isolar as referências
df_ref_o0 = df_baseline[df_baseline["opt_level"] == "O0 (Raw)"][["samples", "features", "fit_time"]].rename(columns={"fit_time": "t_ref_o0"})
df_ref_o3 = df_baseline[df_baseline["opt_level"] == "O3 (Opt)"][["samples", "features", "fit_time"]].rename(columns={"fit_time": "t_ref_o3"})

# Mesclar com os dados paralelos
df_analysis = pd.merge(df_parallel, df_ref_o0, on=["samples", "features"])
df_analysis = pd.merge(df_analysis, df_ref_o3, on=["samples", "features"])

# Calcular as métricas de Speedup
df_analysis["Speedup (vs O0)"] = df_analysis["t_ref_o0"] / df_analysis["fit_time"]
df_analysis["Speedup (vs O3)"] = df_analysis["t_ref_o3"] / df_analysis["fit_time"]

# Substituir divisões por zero ou timeouts por NaN
df_analysis.replace([np.inf, -np.inf], np.nan, inplace=True)

# Melt para o Altair
df_plot = df_analysis.melt(
    id_vars=["problem_size", "threads", "samples", "features"],
    value_vars=["Speedup (vs O0)", "Speedup (vs O3)"],
    var_name="Referência",
    value_name="Speedup",
)

# Gráfico Facetado
speedup_chart = (
    alt.Chart(df_plot)
    .mark_line(point=True)
    .encode(
        x=alt.X('threads:Q', title='Número de Threads'),
        y=alt.Y('Speedup:Q', scale=alt.Scale(type='log'), title='Speedup (Escala Log)'),
        color=alt.Color('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
        tooltip=['problem_size', 'threads', 'Speedup'],
    )
    .properties(width=350, height=400, title='Escalabilidade Paralela ')
    .facet(column=alt.Column('Referência:N', title='Baseline de Referência'))
    .resolve_scale(y='independent')
)

display(speedup_chart)

In [ ]:
hpc_para_targets = ['CPI Rate', 'Memory Bound', 'Cache Bound']
para_hpc_list = []

for _, row in df_parallel.iterrows():
    ds_id = row['dataset'].replace('.csv', '')
    t = int(row['threads'])
    
    fname = f"hpc_{ds_id}_t{t}.csv"
    fpath = os.path.join(DIR_PARA_VTUNE, fname)
    
    m_data = get_hpc_metrics_isolated(fpath, hpc_para_targets)
    
    entry = {
        'problem_size': row['problem_size'],
        'threads': t,
        'samples': row['samples'],
        'features': row['features'],
        **m_data
    }
    para_hpc_list.append(entry)

df_hpc_para = pd.DataFrame(para_hpc_list)

# Gráficos HPC
mem_chart = alt.Chart(df_hpc_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Threads'),
    y=alt.Y('Memory Bound:Q', title='Memory Bound (%)'),
    color=alt.Color('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size', 'threads', 'Memory Bound']
).properties(width=400, height=300, title='Saturação da Memória ')

cpi_chart = alt.Chart(df_hpc_para).mark_line(point=True).encode(
    x=alt.X('threads:Q', title='Threads'),
    y=alt.Y('CPI Rate:Q', title='CPI Rate'),
    color=alt.Color('problem_size:N', sort=problem_order, title='Tamanho do Problema'),
    tooltip=['problem_size', 'threads', 'CPI Rate']
).properties(width=400, height=300, title='Eficiência de Pipeline ')

display((mem_chart | cpi_chart).resolve_scale(y='independent'))